# 🚀 Ferni Router Training (Google Colab)

Train the FTIS tool router with SOTA data on a free T4 GPU.

**Expected time: ~30 minutes**

## Setup
1. Go to Runtime → Change runtime type → GPU (T4)
2. Run all cells
3. Download the model when done

In [ ]:
# Install dependencies
!pip install -q transformers peft datasets scikit-learn accelerate

In [ ]:
# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Upload training data
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
print("Upload train.jsonl, validation.jsonl, and label_map.json:")
uploaded = files.upload()

for name in uploaded.keys():
    os.rename(name, f'data/{name}')
    print(f"  Saved: data/{name}")

In [ ]:
import json
import torch
import torch.nn as nn
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import f1_score
import numpy as np

# Config
MODEL = "Qwen/Qwen2.5-0.5B"
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 2
LEARNING_RATE = 3e-5

In [ ]:
# Load data
def load_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(DATA_DIR / "train.jsonl")
val_data = load_jsonl(DATA_DIR / "validation.jsonl")

with open(DATA_DIR / "label_map.json", 'r') as f:
    label_map = json.load(f)

NUM_LABELS = len(label_map)
print(f"📊 Train: {len(train_data):,}, Val: {len(val_data):,}, Labels: {NUM_LABELS}")

In [ ]:
# Convert to multi-label format
def convert_to_multilabel(data, label_map, num_labels):
    processed = []
    for item in data:
        labels = [0.0] * num_labels
        for tool in item.get('selected_tools', []):
            if tool in label_map:
                labels[label_map[tool]] = 1.0
        processed.append({'text': item['query'], 'labels': labels})
    return processed

train_processed = convert_to_multilabel(train_data, label_map, NUM_LABELS)
val_processed = convert_to_multilabel(val_data, label_map, NUM_LABELS)

train_dataset = Dataset.from_list(train_processed)
val_dataset = Dataset.from_list(val_processed)

In [ ]:
# Load model and tokenizer
print(f"🧠 Loading {MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)
model.config.pad_token_id = tokenizer.pad_token_id

# Add LoRA
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1, bias="none",
    task_type=TaskType.SEQ_CLS,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Tokenize
def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
val_dataset = val_dataset.map(tokenize_fn, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    
    top1_indices = np.argmax(probs, axis=1)
    label_indices = np.argmax(labels, axis=1)
    top1_acc = (top1_indices == label_indices).mean()
    
    top3_indices = np.argsort(probs, axis=1)[:, -3:]
    top3_acc = np.array([label_indices[i] in top3_indices[i] for i in range(len(label_indices))]).mean()
    
    preds = (probs > 0.5).astype(int)
    f1 = f1_score(labels, preds, average='micro', zero_division=0)
    
    return {'f1': f1, 'top1_acc': top1_acc, 'top3_acc': top3_acc}

In [ ]:
# Train
OUTPUT_DIR.mkdir(exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    remove_unused_columns=False,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("🏋️ Starting training...")
trainer.train()

In [ ]:
# Evaluate
print("📊 Final evaluation:")
results = trainer.evaluate()
print(f"   F1: {results['eval_f1']:.4f}")
print(f"   Top-1: {results['eval_top1_acc']:.4f}")
print(f"   Top-3: {results['eval_top3_acc']:.4f}")

In [ ]:
# Save model
import shutil

final_dir = OUTPUT_DIR / "final"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
shutil.copy(DATA_DIR / "label_map.json", final_dir / "label_map.json")

print(f"✅ Model saved to {final_dir}")

In [ ]:
# Download model
!zip -r ferni-router-model.zip output/final/
files.download('ferni-router-model.zip')
print("📥 Download complete! Extract and copy to models/ferni-router-1069/")